# Tutorials

## 🚀 Quick start - install

To install `covmats`, the easiest way is through `pip`:

```bash
    pip install covmats
```
Or alternatively using `conda`

```bash
    conda install covmats
```

You might also clone the repository and install from source

```bash
    pip install -e .
```

Once the installation is done, we can start using covariance matrices. There are various representations:
- CovarianceMatrix
- CovViaDiag
- CovViaDense
- CovViaCholesky
- CovViaEigendecomposition
- CovViaEnsemble
- CovViaFFT
- CovViaPrecision
- CovViaSparseCholesky
- CovViaSparsePrecision


Convention, Q for the precision and Sigma for 

- Let's start with the dense version

In [1]:
import scipy as sp
import covmats

## Diagonal matrix

Let's start with the simple case of a diagonal matrix. 

In [2]:
import numpy as np

d = [1, 2, 3]
A33 = np.diag(d)  # a diagonal covariance matrix
x = [4, -2, 5]  # a point of interest
dist = sp.stats.multivariate_normal(mean=[0, 0, 0], cov=A33)
dist.pdf(x)

np.float64(4.9595685102808205e-08)

It is compatible with the stats API from scipy since the base class inherit from `Covariance`.

In [3]:
cov_diag33 = covmats.CovarianceMatrix.from_diagonal(d)
dist = sp.stats.multivariate_normal(mean=[0, 0, 0], cov=cov_diag33)
dist.pdf(x)

array([4.38559708e-12, 3.37164352e-07, 1.43365931e-05])

In [4]:
cov_diag33.rank, cov_diag33.log_pdet, cov_diag33.get_trace()

(np.int64(3), np.float64(1.791759469228055), 6.0)

- It also behaves as a Linearoperator, supporting matrix-vector, matrix-matrix and solve operations

In [5]:
v3 = np.array([1.0, 2.0, 3.0])
cov_diag33 @ v3

array([1., 4., 9.])

In [6]:
np.linalg.inv(A33) @ v3

array([1., 1., 1.])

In [7]:
cov_diag33.solve(v3)

array([1., 1., 1.])

In [8]:
# product with a matrix (3, 2)
V32 = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]]).T
cov_diag33 @ V32

array([[1., 1.],
       [4., 4.],
       [9., 9.]])

In [9]:
cov_diag33.solve(V32)

array([[1., 1.],
       [1., 1.],
       [1., 1.]])

In [10]:
cov_diag33.whiten(v3)

array([1.        , 1.41421356, 1.73205081])

In [11]:
cov_diag33.whiten(V32).shape

(3, 2)

## Dense covariance matrix

With a dense covariance matrix, it is possible to compute the rank and the determinant directly.

In [12]:
cov_dense33 = covmats.CovarianceMatrix.from_dense(A33)
cov_dense33.rank, cov_dense33.log_pdet, cov_dense33.get_trace()

(np.int64(3), np.float64(1.791759469228055), 6)

It is also possible to perform classic operations such as matrix-vector multiplications, matrix-matrix or even solving systems

In [13]:
v3 = np.array([1.0, 2.0, 3.0])
cov_dense33 @ v3, cov_dense33.solve(v3)

(array([1., 4., 9.]), array([1., 1., 1.]))

In [14]:
# product with a matrix (3, 2)
V32 = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]]).T
cov_dense33 @ V32, cov_dense33.solve(V32)

(array([[1., 1.],
        [4., 4.],
        [9., 9.]]),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]]))

How ever, other operations such as whitthening, etc. requires to decompose the matrix

Or for an another example.

In [15]:
rng = np.random.default_rng(2026)
n = 5
A55 = rng.random(size=(n, n))
A55 = A55 @ A55.T  # make the covariance symmetric positive definite
x = rng.random(size=n)

cov_dense55 = covmats.CovarianceMatrix.from_dense(A55)
cov_dense55.rank, cov_dense55.log_pdet, cov_dense55.get_trace()

(np.int64(5), np.float64(-10.385086971402448), 8.160290791028103)

In [16]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_dense55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [17]:
cov_dense55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [18]:
V52 = np.tile(v5.reshape(-1, 1), 2)
cov_dense55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [19]:
np.linalg.inv(cov_dense55.todense()) @ V52

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

- The output should be identical

In [20]:
cov_dense55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

## Cholesky

In [21]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
L = np.linalg.cholesky(A55)
cov_cho55 = covmats.CovarianceMatrix.from_cholesky(L)
np.allclose(cov_cho55.log_pdet, np.linalg.slogdet(A55)[-1])
cov_cho55.rank, cov_cho55.log_pdet, cov_cho55.get_trace()

(np.int64(5), np.float64(-10.38508697140242), 8.160290791028103)

In [22]:
dist = sp.stats.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_cho55)
dist.pdf(x)

np.float64(0.000619698809563932)

In [23]:
x = np.random.default_rng(676878).normal(size=5)
res = cov_cho55.whiten(x)
ref = sp.linalg.solve_triangular(cov_cho55._factor, x, lower=True)
np.allclose(res, ref)

True

In [24]:
np.testing.assert_allclose(cov_cho55.colorize(cov_cho55.whiten(x)), x)

In [25]:
res = cov_cho55.colorize(x)
res

array([0.41404402, 0.4899987 , 0.32315677, 0.33313679, 0.84757021])

In [26]:
cov_cho55.colorize(np.random.default_rng(676878).normal(size=5))

array([0.41404402, 0.4899987 , 0.32315677, 0.33313679, 0.84757021])

In [27]:
cov_cho55.shape == (5, 5)

True

In [28]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_cho55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [29]:
cov_cho55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [30]:
cov_cho55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [31]:
cov_cho55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [32]:
cov_cho55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

## Working with precision

It is also possible to work with the inverse of the covariance matrix, namely the precision matrix

In [33]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
cov_Q55 = covmats.CovarianceMatrix.from_precision(sp.linalg.inv(A55))
# Sanity checks
np.allclose(cov_Q55.log_pdet, np.linalg.slogdet(A55)[-1])
np.testing.assert_allclose(cov_Q55.todense(), A55)
cov_Q55.rank, cov_Q55.log_pdet, cov_Q55.get_trace()

(np.int64(5), np.float64(-10.385086971403341), 8.160290791021305)

In [34]:
dist = sp.stats.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_Q55)
dist.pdf(x)

np.float64(3.3277799724023296e-241)

In [35]:
x = np.random.default_rng(676878).normal(size=5)

# This is not True => need to check if this is correct ???
np.allclose(cov_Q55.whiten(x), cov_cho55.whiten(x))

False

In [36]:
np.testing.assert_allclose(cov_Q55.colorize(cov_Q55.whiten(x)), x)

In [37]:
cov_Q55.whiten(x)

array([31.80930111,  5.58426935, -7.8710699 ,  0.81242828,  1.73424998])

In [38]:
cov_Q55.shape == (5, 5)

True

In [39]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_Q55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [40]:
cov_Q55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [41]:
sp.sparse.csc_array([[1.0, 1.0]]).todense()

array([[1., 1.]])

In [42]:
cov_Q55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [43]:
cov_Q55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [44]:
cov_Q55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

## Eigen 

Eigen is great ! It can be generated from a dense matrix

In [45]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
cov_eig55 = covmats.eigen_factorize_cov_mat(cov_Q55, n_pc=4, random_state=376)
# Sanity checks
np.allclose(cov_eig55.log_pdet, np.linalg.slogdet(A55)[-1])
np.testing.assert_allclose(cov_eig55.todense(), A55, rtol=0.001)
cov_eig55.rank, cov_eig55.log_pdet, cov_eig55.get_trace()

(np.int64(4), np.float64(-3.9847087229608347), 8.15862986211073)

In [46]:
dist = sp.stats.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_eig55)
dist.pdf(x)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1 is different from 5)

In [ ]:
x = np.random.default_rng(676878).normal(size=5)

# This is not True => need to check if this is correct ???
np.allclose(cov_eig55.whiten(x), cov_Q55.whiten(x))

ValueError: operands could not be broadcast together with shapes (4,) (5,) 

In [ ]:
cov_eig55.whiten(x)

array([-0.32871033,  2.49980091, -2.60598443, -8.30678963])

In [ ]:
(cov_eig55._v.T / np.sqrt(cov_eig55._w)) @ x

array([-0.32871033,  2.49980091, -2.60598443, -8.30678963])

In [ ]:
(cov_eig55._v.T * (1.0 / np.sqrt(cov_eig55._w + eps))).shape

(4, 5)

In [ ]:
x.shape

(5,)

In [ ]:
x

array([ 0.4309492 , -0.03543374, -0.92023751,  2.0224398 ,  1.82102769])

In [ ]:
# Whitening operator (n x n, implicit)
eps = 1e-6
x2 = ((cov_eig55._v.T * (1.0 / np.sqrt(cov_eig55._w + eps))).T @ cov_eig55._v.T) @ z

In [ ]:
x2

array([-0.26345927,  0.24639157, -0.49901551,  1.1380487 ,  2.24964905])

In [ ]:
z = ((cov_eig55._v.T * np.sqrt(cov_eig55._w)).T @ cov_eig55._v.T) @ x

In [ ]:
z

array([-0.80708197,  3.55202411, -5.84634555,  1.7129135 ,  5.63614557])

In [ ]:
(cov_eig55._v.T * np.sqrt(cov_eig55._w))

array([[-8.77371185e-01, -1.35666962e+00, -1.73293123e+00,
        -1.01442716e+00, -9.19486533e-01],
       [ 1.28799005e-01, -3.74734375e-01, -3.74400571e-02,
         2.75321861e-03,  4.97532369e-01],
       [ 3.68961646e-01,  2.27862189e-02,  1.72362264e-02,
        -3.10745016e-01, -7.53362826e-02],
       [-1.05547765e-02, -9.58970277e-02,  1.11975417e-01,
         1.45443402e-03, -6.10776470e-02]])

In [ ]:
np.testing.assert_allclose(cov_Q55.colorize(cov_Q55.whiten(x)), x)

In [ ]:
cov_Q55.whiten(x)

array([31.80930111,  5.58426935, -7.8710699 ,  0.81242828,  1.73424998])

In [ ]:
cov_Q55.shape == (5, 5)

True

In [ ]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_Q55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [ ]:
cov_Q55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [ ]:
cov_Q55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [ ]:
cov_Q55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [ ]:
cov_Q55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

In [ ]:
covmats.eigen_factorize_cov_mat(cov_Q55)

eigen_factorize_cov_mat,
generate_dense_matrix,
get_explained_var,
get_matrix_eigen_factorization,

## SVD

In [ ]:
sp.sparse.linalg.svds()

## Kernel based Covariances

Lib such as gstools, gstlearn, etc. provide kernels

### FFT

### Hirearchical

## Ensemble of realizations

## Sparse precision and cholesky

- Talk about SPDE, large scale applications

# Normal sampling

In [48]:
from covmats._helpers import check_random_state

check_random_state(np.random.default_rng(42)).standard_normal(size=(2, 3, 4))

array([[[ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472],
        [-1.95103519, -1.30217951,  0.1278404 , -0.31624259],
        [-0.01680116, -0.85304393,  0.87939797,  0.77779194]],

       [[ 0.0660307 ,  1.12724121,  0.46750934, -0.85929246],
        [ 0.36875078, -0.9588826 ,  0.8784503 , -0.04992591],
        [-0.18486236, -0.68092954,  1.22254134, -0.15452948]]])

In [80]:
covd = covmats.CovViaDiagonal(np.array([5.0, 10.0, 15.0]))
rng_seed = 42
covd.sample_mvnormal(shape=[2], random_state=rng_seed)

array([[ 1.11068661, -0.43723011,  2.50848692],
       [ 3.40559829, -0.74045799, -0.90680853]])

In [86]:
x = covd.sample_mvnormal(shape=[2, 4], random_state=rng_seed)
x

array([[[ 1.11068661, -0.43723011,  2.50848692],
        [ 3.40559829, -0.74045799, -0.90680853],
        [ 3.53122721,  2.4268417 , -1.81826648],
        [ 1.21320114, -1.46545542, -1.80376358]],

       [[ 0.54104409, -6.05032338, -6.68057804],
        [-1.25731314, -3.20285323,  1.21707469],
        [-2.03040356, -4.46609644,  5.67643327],
        [-0.50485116,  0.21354293, -5.518026  ]]])

In [ ]:
x.shape

(2, 4, 3)

In [92]:
cov_cho = covmats.CovViaCholesky(sp.linalg.cholesky(covd.todense()))
cov_cho.sample_mvnormal(shape=[2], random_state=rng_seed)

array([[ 1.11068661, -0.43723011,  2.50848692],
       [ 3.40559829, -0.74045799, -0.90680853]])

In [93]:
cov_cho.sample_mvnormal(shape=[2, 4], random_state=rng_seed)

array([[[ 1.11068661, -0.43723011,  2.50848692],
        [ 3.40559829, -0.74045799, -0.90680853],
        [ 3.53122721,  2.4268417 , -1.81826648],
        [ 1.21320114, -1.46545542, -1.80376358]],

       [[ 0.54104409, -6.05032338, -6.68057804],
        [-1.25731314, -3.20285323,  1.21707469],
        [-2.03040356, -4.46609644,  5.67643327],
        [-0.50485116,  0.21354293, -5.518026  ]]])